In [8]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import IFrame, display

combined_path = Path.cwd() / "combined_hourly_daily_clean.csv"
df = pd.read_csv(combined_path, parse_dates=["date"])

if "period" not in df.columns:
    df["period"] = df["date"].dt.year.apply(lambda y: "Before 2020" if y < 2020 else "After 2020")

df['group'] = df['station'].str.replace('_',' ').str.replace('airport','',case=False).str.strip() + " — " + df['period']

desired_order = [
    "Dublin — Before 2020",
    "Dublin — After 2020",
    "Cork — Before 2020",
    "Cork — After 2020"
]
present = [g for g in desired_order if g in df['group'].unique()]
df['group'] = pd.Categorical(df['group'], categories=present, ordered=True)

group_means = df.groupby('group', dropna=False)['rain'].mean().reindex(present)

fig = px.violin(
    df,
    x='group',
    y='rain',
    color='group',
    category_orders={'group': present},
    box=True,
    points='outliers',
    width=1000,
    height=600
)

fig.update_layout(showlegend=False)
fig.update_yaxes(title_text="Daily rainfall (mm)", rangemode='tozero')
fig.update_xaxes(title_text="Station & Period")
fig.update_layout(
    title_text="Daily Rainfall — Dublin vs Cork (Before vs After 2020)",
    title_x=0.5,
    template="plotly_white",
    margin=dict(l=60,r=40,t=90,b=60)
)

for grp, mean_val in group_means.items():
    if not pd.isna(mean_val):
        fig.add_trace(go.Scatter(
            x=[grp],
            y=[mean_val],
            mode='markers+text',
            marker=dict(color='black', size=8),
            text=[f"Mean {mean_val:.2f} mm"],
            textposition="top center",
            showlegend=False,
            hoverinfo='skip'
        ))

fig.add_annotation(
    x=0.5, y=1.05,
    xref='paper', yref='paper',
    text="Violin = density; box = quartiles; points = outliers",
    showarrow=False,
    font=dict(size=11, color="grey")
)

out_html = Path.cwd() / "final_interactive_violin.html"
fig.write_html(str(out_html), include_plotlyjs='cdn', full_html=True)

display(IFrame(src=str(out_html), width=1000, height=650))


C:\Users\Hareeshwar\AppData\Local\Temp\ipykernel_31860\2436723916.py:24: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

